In [1]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

# ================================
# AGENT 1: PLANNER
# ================================
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]


# ================================
# AGENT 2: FLIGHT AGENT
# ================================
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


# ================================
# AGENT 3: WEATHER AGENT
# ================================
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result


# RUN
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [12]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)

# ================================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions

# ================================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines

# ================================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient
# ============================================
# 🏥 HOSPITAL MULTI-AGENT SYSTEM (LLM BASED)
# ============================================

# ============================================
# 🏥 HOSPITAL MULTI-AGENT SYSTEM (Groq + LLaMA)
# ============================================

!pip install groq

from groq import Groq
from google.colab import userdata

# ============================================
# 🔑 LOAD API KEY
# ============================================

api_key = userdata.get("groq_api_key")

if api_key is None:
    raise ValueError("❌ Add GROQ_API_KEY in Colab Secrets")

client = Groq(api_key=api_key)


# ============================================
# 🔹 LLM HELPER
# ============================================

def ask_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a medical assistant AI."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5,
        max_tokens=300
    )
    return response.choices[0].message.content


# ============================================
# 🧠 AGENT 1: INTAKE AGENT
# ============================================

def intake_agent(query):
    print("\n🧠 Intake Agent")

    prompt = f"""
    Extract structured patient info from:
    {query}

    Return ONLY:
    Symptoms:
    Age:
    Medical History:
    """

    result = ask_llm(prompt)
    print("Output:", result)

    return result, ["diagnosis", "treatment", "decision"]


# ============================================
# 🔬 AGENT 2: DIAGNOSTIC AGENT
# ============================================

def diagnostic_agent(data):
    print("\n🔬 Diagnostic Agent")

    prompt = f"""
    Patient data:
    {data}

    Identify:
    - Possible diseases
    - Required tests
    """

    result = ask_llm(prompt)
    print("Output:", result)

    return result


# ============================================
# 💊 AGENT 3: TREATMENT AGENT
# ============================================

def treatment_agent(diagnosis):
    print("\n💊 Treatment Agent")

    prompt = f"""
    Diagnosis:
    {diagnosis}

    Suggest:
    - Medicines
    - Lifestyle advice
    - When to consult doctor
    """

    result = ask_llm(prompt)
    print("Output:", result)

    return result


# ============================================
# 🧾 AGENT 4: DECISION AGENT
# ============================================

def decision_agent(data, diagnosis, treatment):
    print("\n🧾 Decision Agent")

    prompt = f"""
    Patient Data: {data}
    Diagnosis: {diagnosis}
    Treatment: {treatment}

    Provide:
    - Final advice
    - Urgency level (Low/Medium/High)
    """

    result = ask_llm(prompt)
    return result


# ============================================
# 🔁 MAIN SYSTEM
# ============================================

def hospital_system(query):
    print("🧑 Patient Query:", query)

    data, plan = intake_agent(query)

    diagnosis = None
    treatment = None
    final = None

    for step in plan:
        if step == "diagnosis":
            diagnosis = diagnostic_agent(data)

        elif step == "treatment":
            treatment = treatment_agent(diagnosis)

        elif step == "decision":
            final = decision_agent(data, diagnosis, treatment)

    return final


# ============================================
# ▶️ RUN
# ============================================

query = input("\nEnter patient symptoms: ")

result = hospital_system(query)

print("\n✅ FINAL RESULT:\n")
print(result)




Enter patient symptoms: i have fever and cough
🧑 Patient Query: i have fever and cough

🧠 Intake Agent
Output: Based on the input "i have fever and cough", I can extract the following structured patient information:

- **Symptoms:**
  - Fever
  - Cough

- **Age:** 
  - Unable to determine (Age was not provided in the input)

- **Medical History:**
  - Unable to determine (Medical history was not provided in the input)

🔬 Diagnostic Agent
Output: **Possible Diseases:**

Based on the symptoms of fever and cough, the following possible diseases can be identified:

1. **Common Cold**: A viral infection that affects the upper respiratory tract, causing symptoms like fever, cough, and runny nose.
2. **Pneumonia**: A bacterial or viral infection that inflames the lungs, causing symptoms like fever, cough, and difficulty breathing.
3. **Bronchitis**: An inflammation of the bronchial tubes, which can be caused by a viral or bacterial infection, leading to symptoms like fever, cough, and wheezi

In [2]:
# Example 2: Manager–Worker Multi-Agent System
# 🧠 Scenario

# “Now instead of fixed flow, we introduce a Manager Agent
# that assigns tasks dynamically to worker

# ================================
# WORKER AGENTS
# ================================
def flight_agent():
    print("[Flight Agent] Working...")
    return [{"airline": "IndiGo", "price": 4500},
            {"airline": "Air India", "price": 5200}]


def weather_agent():
    print("[Weather Agent] Working...")
    return {"condition": "Clear", "temp": 28}


# ================================
# MANAGER AGENT
# ================================
def manager_agent(user_query):
    print("\n[Manager Agent] Analyzing task...")

    tasks = []

    if "trip" in user_query.lower():
        tasks = ["flight", "weather"]

    return tasks


# ================================
# EXECUTION
# ================================
def run_system(user_query):
    print("User Query:", user_query)

    tasks = manager_agent(user_query)

    results = {}

    for task in tasks:
        if task == "flight":
            results["flights"] = flight_agent()

        elif task == "weather":
            results["weather"] = weather_agent()

    # Final decision
    cheapest = min(results["flights"], key=lambda x: x["price"])

    return f"Manager Decision: Book {cheapest['airline']} at ₹{cheapest['price']}"


# RUN
response = run_system("Plan my trip")
print("\nFinal Answer:", response)


User Query: Plan my trip

[Manager Agent] Analyzing task...
[Flight Agent] Working...
[Weather Agent] Working...

Final Answer: Manager Decision: Book IndiGo at ₹4500


In [10]:
# ============================================
# 🏢 CORPORATE MULTI-AGENT SYSTEM (Groq + Colab)
# ============================================

# Install (run once)
!pip install groq

from groq import Groq
from google.colab import userdata

# ============================================
# 🔑 LOAD API KEY
# ============================================

# ⚠️ IMPORTANT: Secret name SAME hona chahiye
api_key = userdata.get("groq_api_key")   # 👈 yahi change kiya hai

if api_key is None:
    raise ValueError("❌ API key not found! Check your Colab Secrets.")

print("✅ API Loaded:", api_key[:10], "...")  # debug

client = Groq(api_key=api_key)


# ============================================
# 🔹 LLM FUNCTION
# ============================================

def ask_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # ✅ updated model
        messages=[
            {"role": "system", "content": "You are a corporate strategy expert."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=300
    )
    return response.choices[0].message.content



# ============================================
# 📊 AGENTS
# ============================================

def market_research_agent(goal):
    print("\n📊 Market Agent")
    return ask_llm(f"Analyze market demand, competition, and customers for: {goal}")


def finance_agent(goal):
    print("\n💰 Finance Agent")
    return ask_llm(f"Analyze investment, ROI, pricing for: {goal}")


def operations_agent(goal):
    print("\n⚙️ Operations Agent")
    return ask_llm(f"Analyze supply chain and logistics for: {goal}")


def legal_agent(goal):
    print("\n⚖️ Legal Agent")
    return ask_llm(f"Analyze regulations and risks for: {goal}")


def hr_agent(goal):
    print("\n👥 HR Agent")
    return ask_llm(f"Analyze hiring and workforce needs for: {goal}")


# ============================================
# 👩‍💼 MANAGER
# ============================================

def manager_agent(goal):
    print("\n👩‍💼 Manager Planning...")

    plan_text = ask_llm(f"""
    Goal: {goal}
    Choose needed departments from:
    market, finance, operations, legal, hr
    Return comma-separated.
    """)

    print("🧠 Plan:", plan_text)

    plan = plan_text.lower().split(",")

    results = {}

    for step in plan:
        step = step.strip()

        if "market" in step:
            results["market"] = market_research_agent(goal)

        elif "finance" in step:
            results["finance"] = finance_agent(goal)

        elif "operations" in step:
            results["operations"] = operations_agent(goal)

        elif "legal" in step:
            results["legal"] = legal_agent(goal)

        elif "hr" in step:
            results["hr"] = hr_agent(goal)

    return results


# ============================================
# 📢 FINAL DECISION
# ============================================

def decision_agent(goal, reports):
    print("\n📢 Final Decision")

    return ask_llm(f"""
    Goal: {goal}
    Reports: {reports}

    Give:
    - Launch or Not
    - Risks
    - Recommendation
    """)


# ============================================
# 🔁 MAIN
# ============================================

def corporate_system(goal):
    print("🎯 Goal:", goal)

    reports = manager_agent(goal)
    final = decision_agent(goal, reports)

    return final


# ============================================
# ▶️ RUN
# ============================================

goal = "Evaluate feasibility of launching Product Y in Asia"

result = corporate_system(goal)

print("\n✅ FINAL STRATEGY:\n")
print(result)

✅ API Loaded: gsk_fGhb5d ...
🎯 Goal: Evaluate feasibility of launching Product Y in Asia

👩‍💼 Manager Planning...
🧠 Plan: To evaluate the feasibility of launching Product Y in Asia, the following departments would be needed:

market, finance, operations, legal

Here's a brief explanation of why each department is necessary:

- Market: To assess the demand for Product Y in the Asian market, understand the target audience, and gather data on competitors.
- Finance: To analyze the financial implications of launching Product Y in Asia, including investment requirements, potential revenue, and return on investment.
- Operations: To evaluate the logistics of delivering Product Y to the Asian market, including supply chain management, distribution channels, and potential partners.
- Legal: To ensure compliance with local laws and regulations in the Asian countries where Product Y will be launched, including intellectual property protection, data privacy, and tax laws.

These departments will 

In [13]:
# Agent 1: Crisis Coordinator (Broadcaster)
# - Broadcasts: “Data breach detected in customer database. Immediate response required.”
# - Sends this to all other agents simultaneously.

# 🛡️ Agent 2: IT Security Agent
# - Receives broadcast.
# - Responds: “Isolate affected servers, patch vulnerabilities, start forensic analysis.”

# 📞 Agent 3: Communications Agent
# - Receives broadcast.
# - Responds: “Draft internal memo, prepare press release, notify stakeholders.”

# 💰 Agent 4: Finance Agent
# - Receives broadcast.
# - Responds: “Estimate financial impact, allocate emergency funds, review insurance coverage.”

# 👩‍⚖️ Agent 5: Legal Agent
# - Receives broadcast.
# - Responds: “Assess regulatory obligations, prepare compliance reports, advise on liability.”

# 👩‍💼 Agent 6: HR Agent
# - Receives broadcast.
# - Responds: “Brief employees, provide guidance on handling customer queries, ensure morale support.”

# 🧑‍⚖️ Agent 7: Decision Agent (Coordinator)
# - Collects all responses.
# - Integrates into a final crisis response plan:
# “Servers isolated, communications prepared, financial impact assessed, compliance secured, employees briefed.”

# ============================================
# 🚨 CRISIS MANAGEMENT MULTI-AGENT SYSTEM
# (Broadcast Architecture + Groq)
# ============================================

!pip install groq

from groq import Groq
from google.colab import userdata

# ============================================
# 🔑 LOAD API KEY
# ============================================

api_key = userdata.get("groq_api_key")

if api_key is None:
    raise ValueError("❌ Add GROQ_API_KEY in Colab Secrets")

client = Groq(api_key=api_key)

# ============================================
# 🔹 LLM FUNCTION
# ============================================

def ask_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a crisis management expert."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.5,
        max_tokens=300
    )
    return response.choices[0].message.content


# ============================================
# 📡 AGENT 1: CRISIS COORDINATOR (BROADCAST)
# ============================================

def coordinator_agent(event):
    print("\n📡 Crisis Coordinator (Broadcasting...)")

    broadcast_message = f"""
    Crisis Alert:
    {event}

    Immediate response required from all departments.
    """

    return broadcast_message


# ============================================
# 🛡️ IT SECURITY AGENT
# ============================================

def it_security_agent(message):
    print("\n🛡️ IT Security Agent")

    prompt = f"""
    Crisis Message:
    {message}

    Provide IT security response actions.
    """

    return ask_llm(prompt)


# ============================================
# 📞 COMMUNICATIONS AGENT
# ============================================

def communications_agent(message):
    print("\n📞 Communications Agent")

    prompt = f"""
    Crisis Message:
    {message}

    Provide communication strategy.
    """

    return ask_llm(prompt)


# ============================================
# 💰 FINANCE AGENT
# ============================================

def finance_agent(message):
    print("\n💰 Finance Agent")

    prompt = f"""
    Crisis Message:
    {message}

    Provide financial response.
    """

    return ask_llm(prompt)


# ============================================
# ⚖️ LEGAL AGENT
# ============================================

def legal_agent(message):
    print("\n⚖️ Legal Agent")

    prompt = f"""
    Crisis Message:
    {message}

    Provide legal and compliance actions.
    """

    return ask_llm(prompt)


# ============================================
# 👩‍💼 HR AGENT
# ============================================

def hr_agent(message):
    print("\n👩‍💼 HR Agent")

    prompt = f"""
    Crisis Message:
    {message}

    Provide HR response and employee guidance.
    """

    return ask_llm(prompt)


# ============================================
# 🧑‍⚖️ DECISION AGENT
# ============================================

def decision_agent(responses):
    print("\n🧑‍⚖️ Decision Agent")

    prompt = f"""
    Combine the following department responses into a single
    clear crisis response plan:

    {responses}

    Provide structured final plan.
    """

    return ask_llm(prompt)


# ============================================
# 🔁 MAIN SYSTEM (BROADCAST FLOW)
# ============================================

def crisis_system(event):
    print("🚨 Crisis Event:", event)

    # Step 1: Broadcast
    message = coordinator_agent(event)

    # Step 2: Parallel responses
    responses = {}

    responses["IT"] = it_security_agent(message)
    responses["Communications"] = communications_agent(message)
    responses["Finance"] = finance_agent(message)
    responses["Legal"] = legal_agent(message)
    responses["HR"] = hr_agent(message)

    # Step 3: Final decision
    final_plan = decision_agent(responses)

    return final_plan


# ============================================
# ▶️ RUN
# ============================================

event = "Data breach detected in customer database. Immediate response required."

result = crisis_system(event)

print("\n✅ FINAL CRISIS RESPONSE PLAN:\n")
print(result)

🚨 Crisis Event: Data breach detected in customer database. Immediate response required.

📡 Crisis Coordinator (Broadcasting...)

🛡️ IT Security Agent

📞 Communications Agent

💰 Finance Agent

⚖️ Legal Agent

👩‍💼 HR Agent

🧑‍⚖️ Decision Agent

✅ FINAL CRISIS RESPONSE PLAN:

**Crisis Management Plan: Data Breach Response**

**I. Incident Response Team**

* **IT Security Response Plan:**
 + **Incident Response Team:**
  - IT Security Manager (Lead)
  - Network Administrator
  - Database Administrator
  - Incident Response Specialist
* **Crisis Management Report:**
 + **Crisis Management Team:**
  - Crisis Manager (Lead)
  - IT Team Lead
  - Communications Team Lead
  - Financial Team Lead
* **Crisis Management Plan: Data Breach Response**
 + **Initial Response (Within 1 hour)**
  - Activate Incident Response Team
  - Assemble a team of experts, including IT, security, compliance, communications, and legal representatives
  - Assign a crisis manager to oversee the response efforts

**II. I

In [14]:
# Scenario: Corporate Product Launch Broadcast

# Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
# “Product X launch in Q3, target market North America, budget $5M.”

# 📢 Coordinator Agent (Broadcaster)
# - Sends the launch announcement to all departments simultaneously.
# - This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

# 📈 Marketing Agent
# - Receives the broadcast.
# - Responds with a marketing strategy: campaigns, channels, and positioning.

# 💰 Finance Agent
# - Receives the broadcast.
# - Responds with budget allocation and ROI forecasts.

# 🏭 Operations Agent
# - Receives the broadcast.
# - Responds with production and supply chain actions.

# 👩‍⚖️ Legal Agent
# - Receives the broadcast.
# - Responds with compliance checks and contract actions.

# 👩‍💼 HR Agent
# - Receives the broadcast.
# - Responds with staffing and training plans.

# 🧑‍⚖️ Decision Agent
# - Collects all responses.
# - Integrates them into a Final Corporate Launch Plan.
# - In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.

# ============================================
# 🚀 CORPORATE PRODUCT LAUNCH SYSTEM (ASYNC)
# ============================================

!pip install groq nest_asyncio

import asyncio
import nest_asyncio
from groq import Groq
from google.colab import userdata

nest_asyncio.apply()  # for Colab async support

# ============================================
# 🔑 API KEY
# ============================================

api_key = userdata.get("groq_api_key")

if api_key is None:
    raise ValueError("❌ Add GROQ_API_KEY in Colab Secrets")

client = Groq(api_key=api_key)

# ============================================
# 🔹 LLM FUNCTION (ASYNC WRAPPER)
# ============================================

async def ask_llm(prompt):
    loop = asyncio.get_event_loop()

    response = await loop.run_in_executor(
        None,
        lambda: client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "You are a corporate strategy expert."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.6,
            max_tokens=300
        )
    )

    return response.choices[0].message.content


# ============================================
# 📢 COORDINATOR (BROADCAST)
# ============================================

async def coordinator_agent(event):
    print("\n📢 Coordinator Broadcasting...")

    message = f"""
    Product Launch Announcement:
    {event}

    Prepare your department response.
    """

    return message


# ============================================
# 📈 MARKETING AGENT
# ============================================

async def marketing_agent(msg):
    print("📈 Marketing Agent running...")
    return await ask_llm(f"Create marketing strategy for:\n{msg}")


# ============================================
# 💰 FINANCE AGENT
# ============================================

async def finance_agent(msg):
    print("💰 Finance Agent running...")
    return await ask_llm(f"Provide budget allocation & ROI for:\n{msg}")


# ============================================
# 🏭 OPERATIONS AGENT
# ============================================

async def operations_agent(msg):
    print("🏭 Operations Agent running...")
    return await ask_llm(f"Plan production & supply chain for:\n{msg}")


# ============================================
# ⚖️ LEGAL AGENT
# ============================================

async def legal_agent(msg):
    print("⚖️ Legal Agent running...")
    return await ask_llm(f"Check compliance & legal risks for:\n{msg}")


# ============================================
# 👩‍💼 HR AGENT
# ============================================

async def hr_agent(msg):
    print("👩‍💼 HR Agent running...")
    return await ask_llm(f"Plan hiring & training for:\n{msg}")


# ============================================
# 🧑‍⚖️ DECISION AGENT
# ============================================

async def decision_agent(responses):
    print("\n🧑‍⚖️ Decision Agent integrating...")

    prompt = f"""
    Combine the following department outputs into a structured
    corporate product launch plan:

    {responses}

    Provide:
    - Summary
    - Key strategies
    - Risks
    - Final recommendation
    """

    return await ask_llm(prompt)


# ============================================
# 🔁 MAIN SYSTEM (ASYNC BROADCAST)
# ============================================

async def corporate_launch_system(event):
    print("🚀 Event:", event)

    # Step 1: Broadcast
    message = await coordinator_agent(event)

    # Step 2: Parallel execution
    results = await asyncio.gather(
        marketing_agent(message),
        finance_agent(message),
        operations_agent(message),
        legal_agent(message),
        hr_agent(message)
    )

    responses = {
        "Marketing": results[0],
        "Finance": results[1],
        "Operations": results[2],
        "Legal": results[3],
        "HR": results[4],
    }

    # Step 3: Final decision
    final = await decision_agent(responses)

    return final


# ============================================
# ▶️ RUN
# ============================================

event = "Product X launch in Q3, target market North America, budget $5M"

result = asyncio.run(corporate_launch_system(event))

print("\n✅ FINAL CORPORATE LAUNCH PLAN:\n")
print(result)

🚀 Event: Product X launch in Q3, target market North America, budget $5M

📢 Coordinator Broadcasting...
📈 Marketing Agent running...
💰 Finance Agent running...
🏭 Operations Agent running...
⚖️ Legal Agent running...
👩‍💼 HR Agent running...

🧑‍⚖️ Decision Agent integrating...

✅ FINAL CORPORATE LAUNCH PLAN:

**Summary:**

The corporate product launch plan for Product X involves a comprehensive strategy across various departments, including Marketing, Finance, Operations, Legal, and HR. The plan aims to create a successful product launch that meets or exceeds sales projections, while ensuring compliance with regulatory requirements and mitigating potential risks.

**Key Strategies:**

1. **Marketing Strategy:**
	* Target the North American market with a budget of $5M.
	* Allocate 40% of the budget to marketing and advertising, including online and offline advertising, influencer marketing, and event marketing.
	* Develop a lead nurturing campaign to educate and engage potential customers